# Hotel Booking Demand — Data Cleaning
**Milestone:** Data Cleaning

## Objective
Inspect a real hotel-booking dataset, identify quality issues, clean them with column-level reasoning, detect outliers, encode categorical data, and compare the dataset before and after cleaning.

**Dataset:** Hotel Booking Demand (`hotel_bookings.csv`)  
**Source:** Kaggle dataset. Keep the original file unchanged; this notebook creates a cleaned copy.

> In Google Colab, upload `hotel_bookings.csv` when prompted. If running locally, place it in the same folder as this notebook.

## 1. Import libraries and load the raw data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("hotel_bookings.csv")

if not DATA_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError("Place hotel_bookings.csv beside this notebook.")

raw_df = pd.read_csv(DATA_PATH)
df = raw_df.copy()

print("Raw shape:", df.shape)
display(df.head())

## 2. Initial data quality assessment

In [ ]:
raw_shape = raw_df.shape
raw_missing = raw_df.isna().sum().sort_values(ascending=False)
raw_duplicates = raw_df.duplicated().sum()

raw_df.info()
print("\\nMissing values by column:")
display(raw_missing[raw_missing > 0].to_frame("missing_count"))
print("Exact duplicate rows:", raw_duplicates)
display(raw_df.describe(include="all").T)

### Initial findings and cleaning decisions

- `children`: only a few missing values. Impute with the median because it is a numeric guest count and median is robust to skew.
- `country`: missing country is unknown. Use the explicit category `Unknown`; do not guess a country.
- `agent`: missing means no agent was recorded. Use `0` to represent no recorded agent, and document that interpretation.
- `company`: many values are missing because most bookings are not associated with a company. Use `0` as “no company recorded” rather than dropping most rows.
- Exact duplicate rows: remove exact matches after measuring them. The dataset has no booking ID, so exact duplicates are indistinguishable records; this is a practical deduplication rule, not proof that the bookings were the same transaction.
- Outliers: use IQR fences and inspect box plots. Keep plausible extreme bookings; remove only impossible or clearly invalid values. Do not automatically delete every IQR-flagged row.

## 3. Remove exact duplicate rows

In [ ]:
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates().copy()
print("Exact duplicates removed:", duplicates_before)
print("Shape after duplicate removal:", df.shape)

## 4. Handle missing values column by column

In [ ]:
missing_before = df.isna().sum()

# Numeric count: median is robust to skew and only 4 values are missing.
children_median = df["children"].median()
df["children"] = df["children"].fillna(children_median)

# Unknown country is meaningful as an unknown category; do not infer it.
df["country"] = df["country"].fillna("Unknown")

# Missing agent/company IDs are treated as no recorded agent/company.
# Zero is used as a sentinel and is explained in the README.
df["agent"] = df["agent"].fillna(0)
df["company"] = df["company"].fillna(0)

print("Missing values remaining:")
display(df.isna().sum()[df.isna().sum() > 0].to_frame("missing_count"))

## 5. Standardise categorical text

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_cols:
    df[col] = df[col].astype("string").str.strip()

# Standardise the month field to title case.
df["arrival_date_month"] = df["arrival_date_month"].str.title()

# Standardise selected category text fields to a consistent case.
for col in ["hotel", "meal", "market_segment", "distribution_channel",
            "deposit_type", "customer_type", "reservation_status"]:
    df[col] = df[col].str.strip()

display(df[categorical_cols].nunique().sort_values(ascending=False).to_frame("unique_values"))

## 6. Detect and assess numeric outliers (IQR + box plots)

In [ ]:
numeric_for_outliers = [
    "lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "booking_changes",
    "days_in_waiting_list", "adr", "required_car_parking_spaces",
    "total_of_special_requests"
]

outlier_rows = []
for col in numeric_for_outliers:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append({
        "column": col, "Q1": q1, "Q3": q3,
        "lower_fence": lower, "upper_fence": upper,
        "IQR_flagged_count": int(flagged)
    })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

In [ ]:
for col in ["lead_time", "adr", "stays_in_week_nights", "adults"]:
    plt.figure(figsize=(8, 2.8))
    plt.boxplot(df[col].dropna(), vert=False)
    plt.title(f"Box plot — {col}")
    plt.xlabel(col)
    plt.show()

### Outlier decisions

- `lead_time`: high values can represent bookings made far in advance; retain plausible values.
- `adr`: high prices can be genuine premium bookings. Negative ADR is invalid, so remove rows with `adr < 0`; retain high positive prices.
- `stays_in_weekend_nights` and `stays_in_week_nights`: long stays can be genuine; retain unless negative.
- `adults`, `children`, `babies`, parking spaces and special requests: unusually high counts may be rare but possible; retain, while remove records with negative counts.
- `days_in_waiting_list` and `booking_changes`: high values can occur operationally; retain unless negative.

IQR flags are used to identify values for review, not as an automatic deletion rule. This avoids deleting genuine rare bookings.

In [ ]:
# Remove only logically invalid values, not merely large IQR-flagged values.
nonnegative_cols = [
    "lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "booking_changes",
    "days_in_waiting_list", "adr", "required_car_parking_spaces",
    "total_of_special_requests", "previous_cancellations",
    "previous_bookings_not_canceled"
]

invalid_mask = pd.Series(False, index=df.index)
for col in nonnegative_cols:
    invalid_mask |= df[col] < 0

invalid_count = int(invalid_mask.sum())
df = df.loc[~invalid_mask].copy()
print("Rows removed for impossible negative values:", invalid_count)

## 7. Encode categorical variables

In [ ]:
# Binary variable is already 0/1; keep it numeric.
binary_cols = ["is_canceled", "is_repeated_guest"]

# Nominal categories: one-hot encode (no artificial order).
nominal_cols = [
    "hotel", "meal", "country", "market_segment",
    "distribution_channel", "reserved_room_type",
    "assigned_room_type", "deposit_type", "customer_type",
    "reservation_status"
]

# Month is cyclic/seasonal rather than ordinal. One-hot encoding avoids
# implying a linear numeric distance between months.
nominal_cols.append("arrival_date_month")

encoded_df = pd.get_dummies(
    df,
    columns=nominal_cols,
    drop_first=False,
    dtype="int8"
)

print("Encoded shape:", encoded_df.shape)
display(encoded_df.head())

## 8. Before-and-after comparison

In [ ]:
# Compare raw data with the cleaned, unencoded data for interpretable statistics.
comparison = pd.DataFrame({
    "Before": {
        "rows": raw_df.shape[0],
        "columns": raw_df.shape[1],
        "missing_cells": int(raw_df.isna().sum().sum()),
        "exact_duplicate_rows": int(raw_df.duplicated().sum())
    },
    "After cleaning": {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum())
    }
})
display(comparison)

stats_cols = ["lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
              "adults", "children", "adr"]
before_stats = raw_df[stats_cols].describe().T
after_stats = df[stats_cols].describe().T
before_stats.columns = [f"before_{c}" for c in before_stats.columns]
after_stats.columns = [f"after_{c}" for c in after_stats.columns]
display(before_stats.join(after_stats))

## 9. Save cleaned datasets

In [ ]:
df.to_csv("hotel_bookings_cleaned.csv", index=False)
encoded_df.to_csv("hotel_bookings_encoded.csv", index=False)
print("Saved hotel_bookings_cleaned.csv and hotel_bookings_encoded.csv")

## 10. Final summary

This workflow:
1. Assessed missing values, data types and exact duplicates.
2. Removed exact duplicate rows.
3. Imputed missing values with a reasoned strategy for each affected column.
4. Used IQR and box plots to review numeric outliers.
5. Removed only logically invalid negative values; retained plausible extreme bookings.
6. One-hot encoded nominal categories while keeping existing binary flags numeric.
7. Compared dataset shape, missing cells, duplicates and numeric statistics before and after.
8. Exported both cleaned and encoded CSV files.

**Important:** The raw dataset is preserved. The cleaning decisions are documented so the process can be reviewed and reproduced.